# Collecte INSEE : population au carreau de 200 m (Filosofi 2021)

Télécharge les carreaux INSEE (Métropole, Martinique, Réunion), calcule le centre de chaque carreau en lon/lat et charge la table Delta `insee_carreaux_200m`.

**Prérequis** : un Lakehouse attaché comme Lakehouse par défaut du notebook.
Les fichiers bruts sont écrits dans `Files/raw/`, les tables Delta dans `Tables/`.

*Notebook généré depuis `src/ingestion/` par `fabric/build_notebooks.py` : modifier les scripts, pas ce notebook.*

In [ ]:
FORCE = False  # True pour retélécharger le fichier INSEE


In [ ]:
%pip install -q pyproj


## Outils communs

In [ ]:
"""Outils communs à la collecte : chemins, téléchargements HTTP, écriture des tables.

Les mêmes scripts tournent en local (sortie dans data/raw/) et dans un notebook
Microsoft Fabric (sortie dans le Lakehouse attaché : /lakehouse/default/Files/raw/).
"""
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

FABRIC_LAKEHOUSE = Path("/lakehouse/default/Files/raw")
LOCAL_RAW = Path(__file__).resolve().parents[2] / "data" / "raw" if "__file__" in globals() else Path("data/raw")

USER_AGENT = "hackathon-mobility-citiz/1.0 (collecte open data)"
PAN_API = "https://transport.data.gouv.fr/api/datasets"


def raw_dir() -> Path:
    """Dossier racine des données brutes : Lakehouse Fabric si présent, sinon data/raw/."""
    base = Path(os.environ["RAW_DIR"]) if "RAW_DIR" in os.environ else (
        FABRIC_LAKEHOUSE if FABRIC_LAKEHOUSE.parent.exists() else LOCAL_RAW
    )
    base.mkdir(parents=True, exist_ok=True)
    return base


def tables_dir() -> Path:
    """Tables Parquet prêtes à être chargées en tables Delta dans Fabric."""
    d = raw_dir() / "tables"
    d.mkdir(parents=True, exist_ok=True)
    return d


def now_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def http_get(url: str, timeout: int = 60, retries: int = 3, **kwargs) -> requests.Response:
    headers = {"User-Agent": USER_AGENT, **kwargs.pop("headers", {})}
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=timeout, **kwargs)
            r.raise_for_status()
            return r
        except requests.RequestException:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)


def get_json(url: str, timeout: int = 30):
    return http_get(url, timeout=timeout).json()


def download(url: str, dest: Path, force: bool = False) -> Path:
    """Télécharge un fichier en streaming. Ne retélécharge pas s'il existe déjà."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0 and not force:
        print(f"  déjà présent : {dest.name}")
        return dest
    tmp = dest.with_suffix(dest.suffix + ".part")
    with http_get(url, timeout=300, stream=True) as r, open(tmp, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    tmp.rename(dest)
    print(f"  téléchargé : {dest.name} ({dest.stat().st_size / 1e6:.1f} Mo)")
    return dest


def save_json(obj, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(json.dumps(obj, ensure_ascii=False))


def _types_homogenes(df: pd.DataFrame) -> pd.DataFrame:
    """Les flux open data mélangent parfois nombres et textes dans une même colonne :
    on convertit ces colonnes en texte pour que Parquet / Delta les acceptent."""
    for col in df.columns[df.dtypes == object]:
        types = {type(v) for v in df[col].dropna()}
        if len(types) > 1 or types - {str}:
            df[col] = df[col].map(lambda v: None if v is None or v != v else str(v))
        if not types:  # colonne entièrement vide : type "null" refusé par Delta, on la type en texte
            df[col] = df[col].astype("string")
    return df


def write_table(df: pd.DataFrame, name: str, append: bool = False) -> Path:
    """Écrit une table Parquet dans <raw>/tables/<name>.parquet (ou ajoute à l'existante)."""
    path = tables_dir() / f"{name}.parquet"
    df = _types_homogenes(df)
    if append and path.exists():
        df = _types_homogenes(pd.concat([pd.read_parquet(path), df], ignore_index=True))
    df.to_parquet(path, index=False)
    print(f"  table {name} : {len(df):,} lignes -> {path}")
    return path


def pan_catalog() -> list:
    """Catalogue complet du Point d'Accès National transport.data.gouv.fr."""
    return get_json(PAN_API, timeout=120)


## Collecte

In [ ]:
"""Collecte INSEE : population et revenus au carreau de 200 m (Filosofi 2021).

Source : https://www.insee.fr/fr/statistiques/8735162
Couverture : France métropolitaine, Martinique, La Réunion.

Sorties :
  <raw>/insee/Filosofi2021_carreaux_200m_csv.zip   (fichier brut INSEE)
  <raw>/tables/insee_carreaux_200m.parquet        (une ligne par carreau habité)
"""
import re
import zipfile

import pandas as pd


FILOSOFI_URL = "https://www.insee.fr/fr/statistiques/fichier/8735162/Filosofi2021_carreaux_200m_csv.zip"
TERRITOIRES = {"met": "Métropole", "mart": "Martinique", "reun": "La Réunion"}
ID_PATTERN = re.compile(r"CRS(\d+)RES200mN(\d+)E(\d+)")


def _ajouter_coordonnees(df: pd.DataFrame) -> pd.DataFrame:
    """Déduit la projection et le centre du carreau à partir de son identifiant INSEE.

    L'identifiant code le coin sud-ouest du carreau (ex. CRS3035RES200mN2029400E4259000) ;
    le centre est donc à +100 m sur chaque axe.
    """
    parts = df["idcar_200m"].str.extract(ID_PATTERN)
    df["epsg"] = parts[0].astype(int)
    df["x_centre"] = parts[2].astype(float) + 100
    df["y_centre"] = parts[1].astype(float) + 100
    try:
        from pyproj import Transformer
    except ImportError:
        print("  pyproj absent : lon/lat non calculées (x_centre / y_centre restent disponibles)")
        return df
    df["lon"], df["lat"] = float("nan"), float("nan")
    for epsg, idx in df.groupby("epsg").groups.items():
        t = Transformer.from_crs(f"EPSG:{epsg}", "EPSG:4326", always_xy=True)
        lon, lat = t.transform(df.loc[idx, "x_centre"].values, df.loc[idx, "y_centre"].values)
        df.loc[idx, "lon"], df.loc[idx, "lat"] = lon, lat
    return df


def collecter(force: bool = False) -> pd.DataFrame:
    print("INSEE Filosofi 2021 - carreaux 200 m")
    zip_path = download(FILOSOFI_URL, raw_dir() / "insee" / "Filosofi2021_carreaux_200m_csv.zip", force=force)

    frames = []
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            code = name.removeprefix("carreaux_200m_").removesuffix(".csv")
            with z.open(name) as f:
                df = pd.read_csv(f, dtype={"idcar_200m": str, "idcar_1km": str, "idcar_nat": str, "lcog_geo": str})
            df.insert(0, "territoire", TERRITOIRES.get(code, code))
            frames.append(df)
            print(f"  {name} : {len(df):,} carreaux")

    df = _ajouter_coordonnees(pd.concat(frames, ignore_index=True))
    df["source"] = "INSEE Filosofi 2021"
    df["date_collecte"] = now_utc()
    write_table(df, "insee_carreaux_200m")
    return df


In [ ]:
collecter(force=FORCE)


## Chargement en tables Delta

In [ ]:
TABLES = ['insee_carreaux_200m']

# Chargement des fichiers Parquet (Files/raw/tables) en tables Delta du Lakehouse par défaut.
# L'écriture passe par l'adresse OneLake (abfss) : le dossier monté /lakehouse/default/Tables
# interrompt les envois de gros fichiers.
from pathlib import Path
import pyarrow.parquet as pq
from deltalake import write_deltalake

try:
    import notebookutils
    ctx = notebookutils.runtime.context
    TABLES_DELTA = (f"abfss://{ctx['defaultLakehouseWorkspaceId']}@onelake.dfs.fabric.microsoft.com/"
                    f"{ctx['defaultLakehouseId']}/Tables" + ("/dbo" if Path("/lakehouse/default/Tables/dbo").exists() else ""))
    OPTIONS = {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}
except ImportError:  # hors Fabric : tables Delta locales, pour tester
    TABLES_DELTA, OPTIONS = str(raw_dir() / "delta"), None

for nom in TABLES:
    source = raw_dir() / "tables" / f"{nom}.parquet"
    if not source.exists():
        print(f"{nom} : absent, ignoré")
        continue
    write_deltalake(f"{TABLES_DELTA}/{nom}", pq.read_table(source), mode="overwrite",
                    schema_mode="overwrite", storage_options=OPTIONS)
    print(f"{nom} : table Delta créée")
